# Credit Card Fraud Detection

Bu notebook'ta [Kaggle Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) veri seti üzerinde dolandırıcılık tespiti yapılıyor.

**Akış:** EDA → feature engineering → train/validation/test ayrımı → Logistic Regression ve Random Forest pipeline'ları → PR-AUC ile model karşılaştırma → validation setinde threshold seçimi → test setinde son değerlendirme → modelin API için kaydedilmesi.

> Veri setinde sınıflar çok dengesiz (~%0.17 fraud). Bu yüzden accuracy yerine **PR-AUC (Average Precision)**, precision ve recall kullanılıyor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
DATA_PATH = "../data/creditcard.csv"  

## 1. Veriyi yükleme

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
print("Toplam eksik değer:", df.isnull().sum().sum())
df.info()

## 2. Keşifsel veri analizi (EDA)
### 2.1 Sınıf dağılımı

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(f"Normal (0): {class_counts[0]:,} (%{class_pct[0]:.3f})")
print(f"Fraud  (1): {class_counts[1]:,} (%{class_pct[1]:.3f})")

plt.figure(figsize=(7, 5))
sns.countplot(x="Class", hue="Class", data=df, palette=["#2ecc71", "#e74c3c"], legend=False)
plt.title("Class Distribution (0: Normal, 1: Fraud)")
plt.xticks([0, 1], ["Normal (0)", "Fraud (1)"])
plt.ylabel("Number of Transactions")
plt.yscale("log")
plt.show()

### 2.2 İşlem tutarı (Amount)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))

ax1.hist(df.loc[df["Class"] == 1, "Amount"], bins=50, color="#e74c3c")
ax1.set_title("Fraud Transactions (Class = 1)")
ax1.set_ylabel("Frequency")
ax1.set_yscale("log")

ax2.hist(df.loc[df["Class"] == 0, "Amount"], bins=50, color="#2ecc71")
ax2.set_title("Normal Transactions (Class = 0)")
ax2.set_xlabel("Amount ($)")
ax2.set_ylabel("Frequency")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

df.groupby("Class")["Amount"].describe()

### 2.3 Zaman → Hour özelliği

`Time`, veri setindeki **ilk işlemden itibaren geçen saniye** sayısını gösteriyor. Veri yaklaşık 2 günü kapsıyor ve başlangıç saati bilinmiyor. Bu yüzden aşağıdaki `Hour` gerçek saati değil, **yaklaşık bir günlük döngüyü** temsil ediyor.

In [ ]:
df["Hour"] = (df["Time"] // 3600) % 24

plt.figure(figsize=(12, 5))
sns.kdeplot(df.loc[df["Class"] == 0, "Hour"], label="Normal (0)", fill=True, color="#2ecc71")
sns.kdeplot(df.loc[df["Class"] == 1, "Hour"], label="Fraud (1)", fill=True, color="#e74c3c")
plt.title("Transaction Density by (Approximate) Hour of Day")
plt.xlabel("Hour (0-23)")
plt.legend()
plt.show()

## 3. Train / validation / test ayrımı

- **Train (%60):** model eğitimi
- **Validation (%20):** model karşılaştırma ve threshold seçimi
- **Test (%20):** sadece en sonda, bir kez kullanılıyor

Ölçekleme pipeline'ın içinde yapılıyor, yani scaler **sadece train verisine** fit ediliyor (data leakage yok).

In [ ]:
FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount", "Hour"]
X = df[FEATURES]
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

for name, ys in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name:<11} n={len(ys):>7,}  fraud={ys.sum():>4}  rate={ys.mean():.5f}")

## 4. Modeller

İki modelde de aynı ön işleme kullanılıyor: `Amount` ve `Hour` için `RobustScaler` (aykırı değerlere dayanıklı). `V1`–`V28` zaten PCA çıktısı olduğu için olduğu gibi geçiyor.

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        [("scale", RobustScaler(), ["Amount", "Hour"])],
        remainder="passthrough",
    )

models = {
    "Logistic Regression": Pipeline([
        ("pre", make_preprocessor()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000,
                                   random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("pre", make_preprocessor()),
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

val_probs = {}
for name, pipe in models.items():
    print(f"Training {name}...")
    pipe.fit(X_train, y_train)
    val_probs[name] = pipe.predict_proba(X_val)[:, 1]

## 5. Validation setinde karşılaştırma (PR-AUC)

Accuracy bu veri setinde yanıltıcı: her şeye "normal" diyen bir model bile %99.8 accuracy alır. Precision-Recall eğrisi ve altındaki alan (Average Precision) çok daha anlamlı bir ölçü.

In [ ]:
plt.figure(figsize=(8, 6))
for name, prob in val_probs.items():
    p, r, _ = precision_recall_curve(y_val, prob)
    ap = average_precision_score(y_val, prob)
    plt.plot(r, p, label=f"{name} (AP = {ap:.3f})")

plt.axhline(y_val.mean(), ls="--", c="gray", label=f"Baseline (AP = {y_val.mean():.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Validation)")
plt.legend()
plt.show()

for name, prob in val_probs.items():
    print(f"\n--- {name} (threshold = 0.5, validation) ---")
    pred = (prob >= 0.5).astype(int)
    print(confusion_matrix(y_val, pred))
    print(classification_report(y_val, pred, digits=3))

## 6. Threshold seçimi (validation setinde)

Varsayılan 0.5 threshold'u, recall'u düşük tutabiliyor. Threshold'u **validation setinde F1'i en üst düzeye çıkaracak şekilde** seçiyoruz. Test setine hâlâ dokunulmadı.

> Gerçek bir senaryoda threshold, iş maliyetine göre seçilir (kaçırılan bir fraud'un maliyeti ile yanlış alarmın maliyeti karşılaştırılır).

In [ ]:
best_name = max(val_probs, key=lambda n: average_precision_score(y_val, val_probs[n]))
best_model = models[best_name]
print("En iyi model (Validation AP):", best_name)

p, r, t = precision_recall_curve(y_val, val_probs[best_name])
f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
best_idx = int(np.argmax(f1))
THRESHOLD = float(t[best_idx])
print(f"Seçilen threshold: {THRESHOLD:.3f}  "
      f"(val precision={p[best_idx]:.3f}, recall={r[best_idx]:.3f}, F1={f1[best_idx]:.3f})")

plt.figure(figsize=(9, 5))
plt.plot(t, p[:-1], label="Precision")
plt.plot(t, r[:-1], label="Recall")
plt.plot(t, f1, label="F1")
plt.axvline(THRESHOLD, ls="--", c="gray", label=f"Threshold = {THRESHOLD:.2f}")
plt.xlabel("Threshold")
plt.title(f"{best_name}: Precision / Recall / F1 vs Threshold (Validation)")
plt.legend()
plt.show()

## 7. Test setinde son değerlendirme

Model ve threshold sabitlendi. Test seti burada **bir kez** kullanılıyor.

In [ ]:
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= THRESHOLD).astype(int)

print(f"Model: {best_name} | Threshold: {THRESHOLD:.3f}")
print(f"Test PR-AUC (Average Precision): {average_precision_score(y_test, test_prob):.3f}\n")

cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test)")
plt.show()

print(classification_report(y_test, test_pred, target_names=["Normal", "Fraud"], digits=3))

## 8. Özellik önemleri

In [ ]:
if best_name == "Random Forest":
    clf = best_model.named_steps["clf"]
    names = best_model.named_steps["pre"].get_feature_names_out()
    names = [n.split("__", 1)[1] for n in names]
    imp = pd.Series(clf.feature_importances_, index=names).sort_values(ascending=False).head(15)
    imp.sort_values().plot.barh(figsize=(8, 6), color="#3498db")
    plt.title("Top 15 Feature Importances (Random Forest)")
    plt.show()

## 9. Modeli kaydetme

Pipeline (scaler + model), threshold ve feature sırası **tek bir dosyada** saklanıyor. Böylece API'de aynı ön işleme ve aynı karar eşiği kullanılıyor.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

bundle = {
    "pipeline": best_model,
    "threshold": THRESHOLD,
    "features": FEATURES,
    "model_name": best_name,
}
joblib.dump(bundle, "../models/fraud_model.joblib")
print("Saved -> models/fraud_model.joblib")